# Pre-processing river water surface elevation (WSE) files from Hydroweb.next to prepare for Altimetric Rating Curves (ARC) fitting
## Tranform the HydroWeb.next virtual station .txt data into a global .csv

_Notebook Authors: Arnaud Cerbelaud, NASA Jet Propulsion Laboratory - California Institute of Technology (April 2024 - April 2025)._ All rights reserved.

In [1]:
#!/usr/bin/env python3
# ******************************************************************************
# ARC_preprocess_Hydroweb.ipynb
# ******************************************************************************

# Purpose:
# Pre-processing water surface elevation files from Hydroweb.next
# Author:
# Arnaud Cerbelaud, 2026

In [10]:
# ******************************************************************************
# Import Python modules
# ******************************************************************************

import os
import pandas as pd
import numpy as np

In [3]:
# ******************************************************************************
# Declaration of variables
# ******************************************************************************
# 1 - data_folder

######
# 1 - data_folder
######
data_folder = ""

# Column name for station identifier: will be the dataset index
ID_VS = 'ID'

In [5]:
# Example file, DO NOT RUN

#BASIN:: AAKOL
#RIVER:: EMEL
#ID:: 0000000009722
#TRIBUTARY OF:: NA
#APPROX. WIDTH OF REACH (m):: 50
#SURFACE OF UPSTREAM WATERSHED (km2):: NA
#RATING CURVE PARAMETERS A,b,Zo such that Q(m3/s) = A[H(m)-Zo]^b:: NA NA NA
#REFERENCE ELLIPSOID:: WGS84
#REFERENCE LONGITUDE:: 82.2061
#REFERENCE LATITUDE:: 46.3663
#REFERENCE DISTANCE (km):: 63
#GEOID MODEL:: EGM2008
#GEOID ONDULATION AT REF POSITION(M.mm):: -52.30
#MISSION(S)-TRACK(S):: S3B-0735
#STATUS:: OPERATIONAL
#VALIDATION CRITERIA:: AUTOMATIC
#MEAN ALTITUDE(M.mm):: 362.52
#MEAN SLOPE (mm/km):: NA
#NUMBER OF MEASUREMENTS IN DATASET:: 69
#FIRST DATE IN DATASET:: 2018-12-15
#LAST DATE IN DATASET:: 2024-11-13
#DISTANCE MIN IN DATASET (km):: 62.4
#DISTANCE MAX IN DATASET (km):: 64.0
#PRODUCTION DATE:: 2024-11-16
#PRODUCT VERSION:: 2.0
#PRODUCT CITATION:: DOI : https://doi.org/10.24400/329360/HYDROWEB_WATER_LEVEL
#SOURCES::
#PRODUCT CONTENT::
#COL 1 : DATE(YYYY-MM-DD)
#COL 2 : TIME(HH:MM)
#COL 3 : ORTHOMETRIC HEIGHT (M) OF WATER SURFACE AT REFERENCE POSITION
#COL 4 : ASSOCIATED UNCERTAINTY(M)
#FIELD SEPARATOR :
#COL 5 : LONGITUDE OF ALTIMETRY MEASUREMENT (deg)
#COL 6 : LATITUDE OF ALTIMETRY MEASUREMENT (deg)
#COL 7 : ELLIPSOIDAL HEIGHT OF ALTIMETRY MEASUREMENT (M)
#COL 8 : GEOIDAL ONDULATION (M) at location [5,6]
#COL 9 : DISTANCE OF ALTIMETRY MEASUREMENT TO REFERENCE POSITION(KM)
#COL 10 : SATELLITE
#COL 11 : ORBIT / MISSION
#COL 12 : GROUND-TRACK NUMBER
#COL 13 : CYCLE NUMBER
#COL 14 : RETRACKING ALGORITHM
#COL 15 : GDR VERSION
################################################################
2018-12-15 15:54 362.72 0.11 : 9999.999 9999.999 310.42 -52.30 9999.999 S3B REP 0735 019 OCOG NA
2019-03-06 15:54 362.49 0.11 : 9999.999 9999.999 310.20 -52.30 9999.999 S3B REP 0735 022 OCOG NA
...

In [6]:
# ******************************************************************************
# Read files
# ******************************************************************************
print('Reading files')

alti_datafiles = []
# Files are actually inside single-file folders
for foldername in os.listdir(data_folder):
    if foldername!='.DS_Store':
        try:
            alti_datafiles.append( data_folder + foldername + '/' +
                               os.listdir(data_folder + foldername)[0]
                             )
        except:
            alti_datafiles.append( data_folder + foldername
                             )

Reading files


In [7]:
# Initializing dataframes

alti_rows = []
alti_u_rows = []

for alti_file in alti_datafiles:

    with open(alti_file, "r") as f:
        content = f.readlines()

    row_data = {}
    row_u_data = {}

    # -----------------------------
    # Read header
    # -----------------------------
    i = 0
    while not content[i].startswith("###"):
        line = content[i]
        key, _, value = line.partition("::")
        key = key[1:]
        value = value.strip()
        row_data[key] = value
        row_u_data[key] = value
        i += 1

    # -----------------------------
    # Read WSE data
    # -----------------------------
    for line in content[i+1:]:
        parts = line.split()
        date = parts[0]   # this is the date of acquisition
        # time = parts[1]   # time of day (can be useful for tide correction)
        wse = np.float32(parts[2])
        wse_u = np.float32(parts[3])

        row_data[date] = wse
        row_u_data[date] = wse_u

    alti_rows.append(row_data)
    alti_u_rows.append(row_u_data)

# -----------------------------
# Build DataFrames once
# -----------------------------
alti_data = pd.DataFrame(alti_rows)
alti_u_data = pd.DataFrame(alti_u_rows)

alti_data.index = alti_data[ID_VS].astype(int)
alti_u_data.index = alti_u_data[ID_VS].astype(int)

# -----------------------------
# Split metadata vs WSE
# -----------------------------
date_mask = pd.to_datetime(alti_data.columns, errors='coerce').notna()

alti_loc_data = alti_data.loc[:, ~date_mask]
wse_data = alti_data.loc[:, date_mask]
wse_u_data = alti_u_data.loc[:, date_mask]

# -----------------------------
# Clean date columns
# -----------------------------
wse_data = wse_data.sort_index(axis=1)
wse_u_data = wse_u_data.sort_index(axis=1)

wse_data.columns = pd.to_datetime(wse_data.columns)
wse_u_data.columns = pd.to_datetime(wse_u_data.columns)

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_52575/3063236002.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_mask = pd.to_datetime(alti_data.columns, errors='coerce').notna()


In [8]:
alti_data   = pd.concat(
    [alti_loc_data, wse_data],
    axis = 1
)
alti_data_u = pd.concat(
    [alti_loc_data, wse_u_data],
    axis = 1
)


In [9]:
### Write to csv

alti_loc_data.to_csv(data_folder + 'vs_database.csv')
alti_data.to_csv(data_folder + 'wse_database.csv')
alti_data_u.to_csv(data_folder + 'wse_u_database.csv')
